# Weather Source
Load weather features and save the final output to CSV.

In [3]:
from pathlib import Path
import pandas as pd

Load regional weather CSV files and convert them into model-ready features.

In [4]:
interval_minutes = 5


def _process_file(df: pd.DataFrame, city: str) -> pd.DataFrame:
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.set_index("datetime").sort_index()
    df = df[~df.index.duplicated(keep="first")]
    df = df.rename(columns={col: f"{str(col).strip().lower().replace(' ', '')}_{city}" for col in df.columns})
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna(axis=1, how="all")
    return df.resample(f"{interval_minutes}min").interpolate(method="time")


def main():
    processed_path = Path("Processed_data/5_weather.csv")

    if processed_path.exists():
        print(f"  {processed_path.name} already exists — skipping.", flush=True)
        return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail()

    sydney    = _process_file(pd.read_csv("Pre_processing/weather_data/NSW weather.csv"), "sydney")
    brisbane  = _process_file(pd.read_csv("Pre_processing/weather_data/QLD weather.csv"), "brisbane")
    melbourne = _process_file(pd.read_csv("Pre_processing/weather_data/VIC weather.csv"), "melbourne")
    adelaide  = _process_file(pd.read_csv("Pre_processing/weather_data/SA weather.csv"),  "adelaide")

    df = pd.concat([sydney, brisbane, melbourne, adelaide], axis=1)
    df.index.name = "SETTLEMENTDATE"
    df.to_csv(processed_path)
    return df.tail()

main()


  5_weather.csv already exists — skipping.


,temp_sydney,feelslike_sydney,dew_sydney,humidity_sydney,precip_sydney,precipprob_sydney,snow_sydney,snowdepth_sydney,windgust_sydney,windspeed_sydney,...,windspeed_adelaide,winddir_adelaide,sealevelpressure_adelaide,cloudcover_adelaide,visibility_adelaide,solarradiation_adelaide,solarenergy_adelaide,uvindex_adelaide,severerisk_adelaide,stations_adelaide
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,,
2025-12-31 22:40:00,19.733333,19.733333,15.8,78.213333,0.0,0.0,0.0,0.0,28.066667,14.566667,...,10.50,170.0,1015.233333,1.1,10.0,0.0,0.0,0.0,10.0,9.467210e+10
2025-12-31 22:45:00,19.725000,19.725000,15.8,78.245000,0.0,0.0,0.0,0.0,27.975000,14.425000,...,10.35,170.0,1015.175000,1.1,10.0,0.0,0.0,0.0,10.0,9.467210e+10
2025-12-31 22:50:00,19.716667,19.716667,15.8,78.276667,0.0,0.0,0.0,0.0,27.883333,14.283333,...,10.20,170.0,1015.116667,1.1,10.0,0.0,0.0,0.0,10.0,9.467210e+10
2025-12-31 22:55:00,19.708333,19.708333,15.8,78.308333,0.0,0.0,0.0,0.0,27.791667,14.141667,...,10.05,170.0,1015.058333,1.1,10.0,0.0,0.0,0.0,10.0,9.467210e+10
2025-12-31 23:00:00,19.700000,19.700000,15.8,78.340000,0.0,0.0,0.0,0.0,27.700000,14.000000,...,9.90,170.0,1015.000000,1.1,10.0,0.0,0.0,0.0,10.0,9.467210e+10
